# ResNet50 Transfer Learning on Fashion-MNIST

IMSA coursework publication copy. Saved outputs were cleared; rerun before citing results.


In [ ]:
from tensorflow import keras
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

## 1. Data Preperation

In [ ]:
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

In [ ]:
import tensorflow as tf

# Normalize
x_train_normalized = x_train.astype('float32')
x_test_normalized = x_test.astype('float32')

# Add channel dimension (28, 28) -> (28, 28, 1)
x_train_expanded = x_train_normalized.reshape(-1, 28, 28, 1)
x_test_expanded = x_test_normalized.reshape(-1, 28, 28, 1)

# Resize to 32x32 (minimum required for ResNet50)
x_train_resized = tf.image.resize(x_train_expanded, (32, 32))
x_test_resized = tf.image.resize(x_test_expanded, (32, 32))

In [ ]:
# Convert grayscale to RGB (1 channel -> 3 channels)
x_train_rgb = np.repeat(x_train_resized.numpy(), 3, axis=-1)
x_test_rgb = np.repeat(x_test_resized.numpy(), 3, axis=-1)
from tensorflow.keras.applications.resnet50 import preprocess_input
x_train_rgb = preprocess_input(x_train_rgb)
x_test_rgb = preprocess_input(x_test_rgb)

# Split
from sklearn.model_selection import train_test_split
x_train_split, x_val, y_train_split, y_val = train_test_split(
    x_train_rgb, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print(f"Training set shape: {x_train_split.shape}")
print(f"Validation set shape: {x_val.shape}")
print(f"Test set shape: {x_test_rgb.shape}")

## 2. Model Selection & Modification


In [ ]:
from keras.applications.resnet50 import ResNet50
from keras.layers import GlobalAveragePooling2D, Dense, Dropout
from keras.models import Model

# Load ResNet50 without the classification head
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Freeze the base model layers
base_model.trainable = False

# New classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(10, activation='softmax')(x)

# Create the final model
model = Model(inputs=base_model.input, outputs=output)

model.summary()

## 3. Training Strategy

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
history = model.fit(
    x_train_split, y_train_split,
    epochs=5,
    batch_size=32,
    validation_data=(x_val, y_val),
    verbose=1
)

## 4. Evaluation & Analysis

In [ ]:
# Report accuracies
train_loss, train_acc = model.evaluate(x_train_split, y_train_split, verbose=1)
val_loss, val_acc = model.evaluate(x_val, y_val, verbose=1)
test_loss, test_acc = model.evaluate(x_test_rgb, y_test, verbose=1)

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Confusion Matrix
y_pred = model.predict(x_test_rgb)
y_pred_classes = np.argmax(y_pred, axis=1)

cm = confusion_matrix(y_test, y_pred_classes)

# Class labels for Fashion MNIST
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Plot confusion matrix
plt.figure(figsize=(12, 10))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=45, ha='right')
plt.yticks(tick_marks, class_names)


plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()


In [ ]:
print(cm) # Numbers on confusion matrix

### Did transfer learning improve performance?
Compared to the past model, the improvement didn't improve. This is probably due to the model being made from only the Resnet50 model plus one additional 256 large dense layer. This made it so the model wasn't able to learn as much during the training, relying only on the information the Resnet50 model already had for most of the time.

### Which architecture did you choose and why?
I chose the ResNet50 model, due to its overall success in image classification. I had just one additional dense layer to help with classification from the Resnet50 model, since I thought the model wouldn't need to much extra layers to classify the images.

## 5. Dicussion

#### Why do pretrained models expect 3-channel images?
Pretrained models expect 3-channel images since those models are typically trained for RGB images which have 3 channels. So the models expect 3-channel, RGB images to be used with them.
#### Why do we freeze layers during transfer learning?
We freeze layers to preserve the information gained from training the model in the past. It also makes it so the large amount of layers in the models won't need to be trained again, saving computation, resulting in only the new layers we added needing to be trained.
#### What challenges arise when using large CNNs on small datasets like Fashion-MNIST
Overfitting can be an issue. If the model is too complex it will learn the dataset rather the patterns in the small 28x28 images. But with this model, it didn't overfit probably due to training only being done on the 1 dense layer.
